# ZD3D — Eikonal Weight Ablation

Sweeps the eikonal loss weight over six values and three random seeds
(18 training runs) to quantify sensitivity to this hyperparameter and select
the value used in `02_train_and_evaluate.ipynb`.

**This benchmark is different from the plain-sphere ones.** The Zalesak initial
field is the CSG difference `max(phi_sphere, -phi_slot)`, which is *not* an
exact signed-distance function near the slot's reentrant edges: it returns the
distance to the nearer face rather than to the edge, so `|grad phi| != 1` there
**in the exact solution itself**. The eikonal term therefore competes with the
initial-condition term near the slot, rather than reinforcing it as it does for
a smooth sphere. Expect the selected weight to be much smaller here, and expect
`w_eik = 1.0` to be actively harmful.

Note that Zalesak is solid-body rotation — a rigid motion — not a deformation
test. It probes preservation of corners and thin features under rotation.

**Runtime:** longer per run than the sphere benchmarks (Fourier features,
RAD/RAR resampling, causal weighting). Set `EST_MIN_PER_RUN` from your first
measured run. Results are appended to CSV after every run and completed runs
are skipped on restart, so a disconnect costs at most one run.

**Outputs:** `ZD3D_eikonal_ablation.csv` with per-snapshot and averaged metrics
for every run, plus a summary table with seed spread.

In [ ]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/ZD3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import zd3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Mounted at /content/drive
device: cuda


In [ ]:
# --- configuration -----------------------------------------------------
W_EIK_GRID   = [0.0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
SEEDS        = [42, 43, 44]

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 10000, 5000
LAYERS, NEURONS = 8, 256
N_FOURIER, B_SCALE = 128, 5.0
T_FINAL      = 2 * np.pi
ADAM_EPOCHS  = 20000        # same budget as the reported run, so the ranking
LBFGS_EPOCHS = 500          # transfers without extrapolation
GRID_N       = 201
N_SNAPSHOTS  = 6

M_CHUNKS, EPS_CAUSAL = 32, 1.0
RAD_FREQ, RAD_CANDIDATES, RAD_K, RAD_C = 1000, 20000, 1.0, 1.0
RAR_FREQ, RAR_CANDIDATES, RAR_ADD      = 1800, 50000, 500

EST_MIN_PER_RUN = 15        # update from your first measured run
CSV_PATH = f'{PROJECT_DIR}/ZD3D_eikonal_ablation.csv'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)
METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs', 'vol_floor']

FIELDS = ['w_eik', 'seed']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['loss_adam', 'loss_final', 'lbfgs_gain', 'n_points', 'minutes']

print(f'{len(W_EIK_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

18 runs, 49 columns per row


In [ ]:
# --- one run -----------------------------------------------------------
def run_one(w_eik, seed):
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS,
                           n_fourier=N_FOURIER, b_scale=B_SCALE)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    sampler = core.AdaptiveSampler(net, X_f, T_FINAL, device,
                                   rad_freq=RAD_FREQ, rad_candidates=RAD_CANDIDATES,
                                   rad_k=RAD_K, rad_c=RAD_C,
                                   rar_freq=RAR_FREQ, rar_candidates=RAR_CANDIDATES,
                                   rar_add=RAR_ADD)
    loss_fn = core.make_loss(net, sampler, X_i, T_FINAL, W_PDE, W_IC, w_eik,
                             m_chunks=M_CHUNKS, eps_causal=EPS_CAUSAL)

    tr = core.train(net, loss_fn, sampler=sampler, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)
    res = core.evaluate(net, grid, times, T_FINAL, device, cache=False)

    row = {'w_eik': w_eik, 'seed': seed}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({'loss_adam': f"{tr['loss_adam']:.6e}",
                'loss_final': f"{tr['loss_final']:.6e}",
                'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
                'n_points': tr['n_points'],
                'minutes': f"{tr['minutes']:.2f}"})

    del net, X_f, X_i, sampler
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) -------------------------------------------------
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['w_eik']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(W_EIK_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * EST_MIN_PER_RUN / 60:.1f} h\n', flush=True)

for k, (wv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] w_eik={wv:g} seed={sd} ...', flush=True)
    row = run_one(wv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"eik={float(row['avg_eik_dev']):.3e}  "
          f"lbfgs_gain={float(row['lbfgs_gain']):.2e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

18 run(s) to go, ~4.5 h

[1/18] w_eik=0 seed=42 ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    relL2=2.4478%  MAPE=8.6212%  eik=2.131e-01  lbfgs_gain=1.39e-08  (14.5 min)
[2/18] w_eik=0 seed=43 ...
    relL2=37.4998%  MAPE=99.9977%  eik=7.759e-01  lbfgs_gain=1.69e-03  (17.4 min)
[3/18] w_eik=0 seed=44 ...
    relL2=2.7760%  MAPE=1.5577%  eik=2.394e-01  lbfgs_gain=9.71e-06  (15.5 min)
[4/18] w_eik=1e-05 seed=42 ...
    relL2=2.3709%  MAPE=7.0310%  eik=2.163e-01  lbfgs_gain=1.52e-08  (14.4 min)
[5/18] w_eik=1e-05 seed=43 ...
    relL2=2.7041%  MAPE=13.6723%  eik=3.111e-01  lbfgs_gain=1.93e-05  (15.8 min)
[6/18] w_eik=1e-05 seed=44 ...
    relL2=2.7027%  MAPE=7.3481%  eik=2.011e-01  lbfgs_gain=8.72e-06  (15.3 min)
[7/18] w_eik=0.0001 seed=42 ...
    relL2=1.6643%  MAPE=6.7744%  eik=1.841e-01  lbfgs_gain=1.36e-08  (14.4 min)
[8/18] w_eik=0.0001 seed=43 ...
    relL2=1.7079%  MAPE=1.1530%  eik=1.556e-01  lbfgs_gain=1.08e-05  (15.8 min)
[9/18] w_eik=0.0001 seed=44 ...
    relL2=1.6503%  MAPE=5.7303%  eik=1.942e-01  lbfgs_gain=1.16e-05  (15.8 min)
[10/18] w_eik=0.001 seed=42 ...
  

## Summary and selection

Decision rule, fixed before the sweep:

1. **Primary** — lowest average relative L₂
2. **Tiebreak 1** — lowest average volume MAPE
3. **Tiebreak 2** — lowest eikonal deviation
4. **Tiebreak 3** — the smaller weight

Two weights count as *tied* when the gap between their means is smaller than
the larger of their two seed standard deviations. Eikonal deviation is only a
second-order tiebreak on purpose: it is essentially the quantity the swept term
minimises, so ranking by it would recover the sweep by construction.

In [5]:
# --- summary -----------------------------------------------------------
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['w_eik']), []).append(r)

stats = {}
print(f"{'w_eik':>8} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'eik dev':>11} | {'n':>2}")
print('-' * 74)
for wv in sorted(agg):
    g = agg[wv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in
         ('rel_L2', 'vol_MAPE', 'eik_dev')}
    stats[wv] = (m, s, len(g))
    print(f"{wv:>8g} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{m['eik_dev']:>11.3e} | {len(g):>2}")

def tied(a, b, key):
    (ma, sa, _), (mb, sb, _) = stats[a], stats[b]
    return abs(ma[key] - mb[key]) < max(sa[key], sb[key])

best = min(stats, key=lambda w: stats[w][0]['rel_L2'])
for cand in sorted(stats, key=lambda w: stats[w][0]['rel_L2'])[1:]:
    if tied(best, cand, 'rel_L2'):
        if not tied(best, cand, 'vol_MAPE'):
            best = min(best, cand, key=lambda w: stats[w][0]['vol_MAPE'])
        elif not tied(best, cand, 'eik_dev'):
            best = min(best, cand, key=lambda w: stats[w][0]['eik_dev'])
        else:
            best = min(best, cand)

ties = sorted(w for w in stats if w != best and tied(best, w, 'rel_L2'))
print('-' * 74)
print(f'selected w_eik = {best:g}')
if ties:
    print('statistically tied on rel L2 with: ' + ', '.join(f'{w:g}' for w in ties))
    print('-> report insensitivity across this range rather than a single value')
print(f"mean L-BFGS gain over all runs: "
      f"{np.mean([float(r['lbfgs_gain']) for r in rows]):.3e}")

   w_eik |          rel L2 (%) |        vol MAPE (%) |     eik dev |  n
--------------------------------------------------------------------------
       0 |   14.2412 ± 16.4469 |   36.7255 ± 44.8330 |   4.095e-01 |  3
   1e-05 |    2.5926 ± 0.1568  |    9.3505 ± 3.0587  |   2.428e-01 |  3
  0.0001 |    1.6742 ± 0.0245  |    4.5526 ± 2.4414  |   1.780e-01 |  3
   0.001 |    1.0141 ± 0.2896  |    7.4286 ± 2.8890  |   8.547e-02 |  3
    0.01 |    1.1806 ± 0.0964  |   16.8426 ± 0.6637  |   5.057e-02 |  3
     0.1 |    1.3920 ± 0.2027  |   18.7144 ± 0.3330  |   4.551e-02 |  3
--------------------------------------------------------------------------
selected w_eik = 0.001
statistically tied on rel L2 with: 0, 0.01
-> report insensitivity across this range rather than a single value
mean L-BFGS gain over all runs: 1.157e-04


## Error growth per snapshot

Per-snapshot averages across seeds, for the time-evolution table/figure.

In [6]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'rel_L2'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

rel_L2 by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    4.0591 |   11.7825 |   15.4299 |   17.1097 |   18.0848 |   18.9811
   1e-05 |    0.2292 |    2.0938 |    3.0790 |    3.1379 |    3.3237 |    3.6919
  0.0001 |    0.2028 |    1.2922 |    1.8899 |    1.8578 |    2.0566 |    2.7456
   0.001 |    0.5466 |    0.7912 |    0.9900 |    1.1401 |    0.9831 |    1.6338
    0.01 |    0.7908 |    1.1252 |    1.1671 |    1.3198 |    1.2264 |    1.4542
     0.1 |    0.8646 |    1.1489 |    1.2183 |    1.4095 |    1.5126 |    2.1983


In [7]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'vol_MAPE'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

vol_MAPE by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |   35.9584 |   36.1646 |   36.6513 |   37.0032 |   36.9810 |   37.5946
   1e-05 |    5.0797 |    8.1866 |    9.6947 |   10.4940 |   10.6352 |   12.0128
  0.0001 |    2.2399 |    3.8414 |    4.3472 |    5.6361 |    5.0776 |    6.1733
   0.001 |    8.6396 |    8.9466 |    8.0334 |    7.2026 |    5.9668 |    5.7829
    0.01 |   14.3355 |   16.7834 |   16.8633 |   17.4869 |   17.8052 |   17.7815
     0.1 |   15.8454 |   18.9314 |   19.1372 |   19.4332 |   19.4566 |   19.4824


In [8]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'eik_dev'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

eik_dev by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.3485 |    0.2917 |    0.3530 |    0.4579 |    0.5001 |    0.5056
   1e-05 |    0.1682 |    0.2106 |    0.2376 |    0.2611 |    0.2870 |    0.2925
  0.0001 |    0.1389 |    0.1620 |    0.1783 |    0.1900 |    0.1956 |    0.2031
   0.001 |    0.1470 |    0.0829 |    0.0674 |    0.0685 |    0.0688 |    0.0782
    0.01 |    0.2057 |    0.0240 |    0.0179 |    0.0173 |    0.0175 |    0.0211
     0.1 |    0.2230 |    0.0090 |    0.0093 |    0.0095 |    0.0108 |    0.0114


In [9]:
# --- per-snapshot table ------------------------------------------------
METRIC = 'l2_abs'      # 'rel_L2' | 'vol_MAPE' | 'eik_dev' | 'l2_abs'

hdr = ' | '.join(f'{lab:>9}' for lab in tlabels)
print(f'{METRIC} by snapshot (mean over seeds)\n')
print(f"{'w_eik':>8} | {hdr}")
print('-' * (11 + 12 * len(tlabels)))
for wv in sorted(agg):
    vals = [np.mean([float(r[f'{METRIC}_t{i}']) for r in agg[wv]])
            for i in range(N_SNAPSHOTS)]
    print(f'{wv:>8g} | ' + ' | '.join(f'{v:>9.4f}' for v in vals))

l2_abs by snapshot (mean over seeds)

   w_eik |         0 |       T/5 |      2T/5 |      3T/5 |      4T/5 |         T
-----------------------------------------------------------------------------------
       0 |    0.0171 |    0.0497 |    0.0651 |    0.0722 |    0.0763 |    0.0801
   1e-05 |    0.0010 |    0.0088 |    0.0130 |    0.0132 |    0.0140 |    0.0156
  0.0001 |    0.0009 |    0.0055 |    0.0080 |    0.0078 |    0.0087 |    0.0116
   0.001 |    0.0023 |    0.0033 |    0.0042 |    0.0048 |    0.0041 |    0.0069
    0.01 |    0.0033 |    0.0047 |    0.0049 |    0.0056 |    0.0052 |    0.0061
     0.1 |    0.0036 |    0.0048 |    0.0051 |    0.0059 |    0.0064 |    0.0093
